# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`. Recon is
then run explicitly against that same `workflow_run_id`, comparing Interface and GL.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for
GL, `ReconClient` for recon).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

## Spark session

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 13:03:20 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/12 13:03:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-14833f68-6b21-475c-b8b0-9087de254d8b;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 78ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## Imports and helpers

In [3]:
from datetime import date

from pyspark.sql import functions as F

from core.logging import configure_logging
from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import POSTGRES_TABLE_LOCATIONS

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from workflow import WorkflowOrchestrator


configure_logging()


def display_df(df):
    display(df.toPandas())

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` —
`GLClient` only knows GL processing, and `ReconClient` only knows recon processing.
`WorkflowOrchestrator` is the thin layer that owns execution/workflow lifecycle across
Foundry and GL and coordinates Foundry → GL. Recon is not yet wired into the
orchestrator (v1 keeps it a standalone `recon.reconcile(workflow_run_id)` call,
run explicitly below once GL has posted).

In [4]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [5]:
workflow_result = orchestrator.run_workflow()

business_dt = pipeline.config.business_dt

# V1 has one producer execution per workflow per output table, so
# workflow_run_id alone is the operational key for every read below.
# gl_run_id (GL's own producer_run_id) is kept only as exact lineage.
workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

2026-09-12 13:03:22,838 | INFO | workflow.orchestrator | Workflow started | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d | dataclass=TRIAL_BALANCE | business_dt=2026-03-31
2026-09-12 13:03:22,844 | INFO | workflow.orchestrator | Foundry pipeline started | run_id=c0f84bba-84ee-428d-a7cd-e160bb1ae68d | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d
2026-09-12 13:03:22,847 | INFO | workflow.orchestrator | Foundry zone started | operation=STAGING | run_id=e9864402-6c3a-40dd-b571-43248efd4d45 | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d
2026-09-12 13:03:28,792 | INFO | workflow.orchestrator | Foundry zone succeeded | operation=STAGING | run_id=e9864402-6c3a-40dd-b571-43248efd4d45 | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d | records=42
2026-09-12 13:03:28,797 | INFO | workflow.orchestrator | Foundry zone started | operation=ENRICHMENT | run_id=91e93372-7d08-4e7c-989a-1e42018a5d5b | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d
2026-09-12 13:03:35,64

2026-09-12 13:03:45,513 | INFO | workflow.orchestrator | GL import succeeded | run_id=bc9dce40-25f3-4af3-bd07-822736e705f0 | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d | received=7 | posted=7 | rejected=0 | source_producer_run_id=ff58950e-7687-4248-b113-5a87f5ff6c3e
2026-09-12 13:03:45,516 | INFO | workflow.orchestrator | Workflow succeeded | workflow_run_id=694af9a4-9c03-419f-a9d6-705ab094677d


## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`workflow_run_id` of the workflow that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`. `PRODUCER_RUN_ID` remains stamped on every row as the exact
producer execution's lineage.

### Source

In [6]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,80000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,10000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,90000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,001000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,90000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,001200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,001200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,001200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,001200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


### Staging

In [7]:
stg_df = repository.read_staging(workflow_run_id)

display_df(stg_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,10000.000000000000,1.000000000000,10000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,USD,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,80000.000000000000,1.000000000000,80000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,20000.000000000000,1.000000000000,20000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,5000.000000000000,1.000000000000,5000.000000000000,DEBIT,694af9a4-9c03-419f-a9d6-705ab094677d,e9864402-6c3a-40dd-b571-43248efd4d45


### Enrichment

In [8]:
enr_df = repository.read_enrichment(workflow_run_id)

display_df(enr_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,101000,201000,101000,001000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,120000,220000,120000,001200,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,USM,TRD,002000,LIABILITY,CREDIT,...,130000,210000,210000,002000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,USM,FIN,004000,REVENUE,CREDIT,...,4100000,4100000,4100000,004000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,USM,FIN,003000,EQUITY,CREDIT,...,310000,310000,310000,003000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,CAM,TRD,001000,ASSET,DEBIT,...,101000,201000,101000,001000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,CAM,TRD,002100,LIABILITY,CREDIT,...,,,,,100001,,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,91e93372-7d08-4e7c-989a-1e42018a5d5b


### Reporting

In [9]:
rpt_df = repository.read_reporting(workflow_run_id)

display_df(rpt_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,101000,201000,101000,001000,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,001000,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,120000,220000,120000,001200,000000,000000,LGAAP1,NFTBAL,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,001200,ASSET,DEBIT,...,,,,,,,,,694af9a4-9c03-419f-a9d6-705ab094677d,656766a0-19b6-4bd9-8c68-24334f84a47a


### Posting

In [10]:
pst_df = repository.read_posting(workflow_run_id)

display_df(pst_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,POSTING_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,USM,TRD,001000,ASSET,...,0E-12,90000.000000000000,80000.000000000000,10000.000000000000,0E-12,90000.000000000000,0E-12,90000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,USM,FIN,001200,ASSET,...,0E-12,20000.000000000000,15000.000000000000,5000.000000000000,0E-12,20000.000000000000,0E-12,20000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,USM,TRD,002000,LIABILITY,...,0E-12,-65000.000000000000,-55000.000000000000,0E-12,-10000.000000000000,-65000.000000000000,0E-12,-65000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,USM,FIN,004000,REVENUE,...,0E-12,-25000.000000000000,-20000.000000000000,0E-12,-5000.000000000000,-25000.000000000000,0E-12,-25000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,USM,FIN,003000,EQUITY,...,0E-12,-20000.000000000000,-20000.000000000000,0E-12,0E-12,-20000.000000000000,0E-12,-20000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,CAM,TRD,001000,ASSET,...,1000.000000000000,51000.000000000000,40000.000000000000,10000.000000000000,0E-12,50000.000000000000,1000.000000000000,51000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,CAM,TRD,002100,LIABILITY,...,-1000.000000000000,-51000.000000000000,-40000.000000000000,0E-12,-10000.000000000000,-50000.000000000000,-1000.000000000000,-51000.000000000000,694af9a4-9c03-419f-a9d6-705ab094677d,528bd3f2-300b-43c7-b06c-6fdb6e119a34


### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` rather than business date/batch.

In [11]:
int_df = repository.read_interface(workflow_run_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,ENTITY_CD,BRANCH_CD,DEPT_CD,GL_ACCOUNT,SUB_ACCOUNT,...,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,1,USMKTS,USNY01,USTRD1,101000,001000,...,GROSS_UP,rec-1,NFM,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
1,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,2,USMKTS,USNY01,USFIN1,120000,001200,...,GROSS_UP,rec-2,NFM,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
2,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,3,USMKTS,USNY01,USTRD1,210000,002000,...,GROSS_UP,rec-3,NFM,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,4,USMKTS,USNY01,USFIN1,4100000,004000,...,GROSS_UP,rec-4,NFM,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,5,USMKTS,USNY01,USFIN1,310000,003000,...,GROSS_UP,rec-5,NFM,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-CAM-...,1,CAMKTS,CATO01,CATRD1,101000,001000,...,GROSS_UP,rec-6,NFM,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
6,694af9a4-9c03-419f-a9d6-705ab094677d,ff58950e-7687-4248-b113-5a87f5ff6c3e,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-CAM-...,2,CAMKTS,CATO01,CATRD1,,,...,GROSS_UP,rec-7,NFM,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31


## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above — selected by `workflow_run_id`, since V1 has one Interface producer execution per
workflow — and stamps its own execution identity onto whatever it writes: `gl.posting`
and `gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's
ID. The reads below select by `workflow_run_id` too, not `gl_run_id`.

### GL Posting

In [12]:
gl_postings = gl.get_postings(workflow_run_id)

display_df(gl_postings)

,GL_POSTING_ID,POSTED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,FOUNDRY_RULE_ID,POSTING_ID,POSTING_STREAM,...,BOOK_CD,SOURCE_CD,CR_DR_IND,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,ac152d68-9596-4142-b3a8-92dd30c051ca,2026-09-12 13:03:39.229332,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-CAM-...,1,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,DR,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
1,e2a3902d-6fb4-4b0c-8505-ef260f6a6127,2026-09-12 13:03:40.979800,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-CAM-...,2,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,CR,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31
2,5679cfd7-717e-4108-adbb-3c97e350fc33,2026-09-12 13:03:41.685246,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,1,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,DR,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,06296678-486a-4267-97ac-787ce9b882ed,2026-09-12 13:03:42.428527,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,2,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,DR,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,dbc7f13e-4973-46bf-a648-64c4269cce65,2026-09-12 13:03:43.269390,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,3,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,CR,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,b0cb961e-b8c6-4973-977b-146af34b5c1c,2026-09-12 13:03:44.368578,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,4,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,CR,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
6,9baeba0b-f6b2-4b9b-b974-c5ed61de0765,2026-09-12 13:03:45.365411,694af9a4-9c03-419f-a9d6-705ab094677d,bc9dce40-25f3-4af3-bd07-822736e705f0,TRIAL_BALANCE,NFTB-91e93372-7d08-4e7c-989a-1e42018a5d5b-USM-...,5,TB-GROSS-UP,PST-260331-260331-528bd3f2-300b-43c7-b06c-6fdb...,GROSS_UP,...,LGAAP1,NFTBAL,CR,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31


### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [13]:
gl_rejections = gl.get_rejections(workflow_run_id)

display_df(gl_rejections)

,GL_REJECTION_ID,REJECTED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,FOUNDRY_RULE_ID,POSTING_ID,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,BUSINESS_DATE,AS_OF_DATE,REJECTION_TYPE,REJECTION_DETAIL


## Recon

`recon.reconcile(workflow_run_id)` reads `interface.trial_balance` and `gl.posting` for
the workflow — the GL side through the real `GLClient` abstraction, not a separate query
— aggregates each side to a balance per `RECON_KEYS` grain (`WORKFLOW_RUN_ID`,
`AS_OF_DATE`, and the nine GL segments plus `ACCOUNTED_CURRENCY`), and persists the
comparison to `recon.result`.

This runs under its own `RECON / RECONCILE` execution, created under the same
`workflow_run_id` being reconciled. That execution's own `run_id` is stamped as
`PRODUCER_RUN_ID` on every row it writes — distinct from `gl_run_id` above, which is GL's
own producer lineage for the postings being compared.

In [14]:
recon_result = recon.reconcile(workflow_run_id)

recon_run_id = recon_result.producer_run_id

print(
    f"result_count={recon_result.result_count} "
    f"break_count={recon_result.break_count} "
    f"producer_run_id={recon_run_id}"
)

result_count=10 break_count=7 producer_run_id=a48af47a-994d-456d-8b7d-16bcfafe9086


In [15]:
recon_df = recon.get_results(workflow_run_id)

display_df(recon_df)

,RECON_RESULT_ID,RECONCILED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,AS_OF_DATE,ENTITY_CD,BRANCH_CD,DEPT_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT
0,fff4da11-e718-4761-a23c-01c1a322ab4e,2026-09-12 13:03:47.107041,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,120000,001200,000000,000000,LGAAP1,NFTBAL,USD,20000.000000000000,20000.000000000000,0E-12
1,dbc5342b-1bc1-4ef9-a77c-2f8a358d383e,2026-09-12 13:03:47.107075,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,990101,990001,000000,000000,LGAAP1,NFTBAL,USD,0E-12,-45000.000000000000,45000.000000000000
2,4fe199b3-8537-4650-9860-a969989d8d43,2026-09-12 13:03:47.107054,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,310000,003000,000000,000000,LGAAP1,NFTBAL,USD,-20000.000000000000,0E-12,-20000.000000000000
3,e0161602-3a56-4b0c-9165-2a3bfb06f733,2026-09-12 13:03:47.107108,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USTRD1,990101,002000,000000,000000,LGAAP1,NFTBAL,USD,0E-12,-65000.000000000000,65000.000000000000
4,bad09dbc-2f3f-4f37-8e41-1528a0763c8b,2026-09-12 13:03:47.107086,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USTRD1,101000,001000,000000,000000,LGAAP1,NFTBAL,USD,90000.000000000000,90000.000000000000,0E-12
5,6a1980e4-59fd-4d6b-a905-b745b80468f4,2026-09-12 13:03:47.107030,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,CAMKTS,CATO01,CATRD1,990201,990001,100001,990003,LGAAP1,NFTBAL,USD,0E-12,-38250.000000000000,38250.000000000000
6,0a6a3f0c-b9cf-49ba-8875-14bd28cd23af,2026-09-12 13:03:47.106993,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,CAMKTS,CATO01,CATRD1,,,100001,,LGAAP1,NFTBAL,USD,-38250.000000000000,0E-12,-38250.000000000000
7,56113b7e-582f-41d9-98ed-8575fceda4ab,2026-09-12 13:03:47.107097,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USTRD1,210000,002000,000000,000000,LGAAP1,NFTBAL,USD,-65000.000000000000,0E-12,-65000.000000000000
8,375cb26f-5b38-4d0e-a3af-e22b12def3c5,2026-09-12 13:03:47.107018,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,CAMKTS,CATO01,CATRD1,101000,001000,000000,000000,LGAAP1,NFTBAL,USD,38250.000000000000,38250.000000000000,0E-12
9,bd5598a3-6a7c-40d0-a3bc-42d2ca52f930,2026-09-12 13:03:47.107065,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,4100000,004000,000000,000000,LGAAP1,NFTBAL,USD,-25000.000000000000,0E-12,-25000.000000000000


### Breaks

Rows with a non-zero `DIFFERENCE_AMOUNT` — expected to be empty for this balanced
synthetic run.

In [16]:
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

display_df(breaks_df)

,RECON_RESULT_ID,RECONCILED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,AS_OF_DATE,ENTITY_CD,BRANCH_CD,DEPT_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT
0,dbc5342b-1bc1-4ef9-a77c-2f8a358d383e,2026-09-12 13:03:47.107075,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,990101,990001,000000,000000,LGAAP1,NFTBAL,USD,0E-12,-45000.000000000000,45000.000000000000
1,4fe199b3-8537-4650-9860-a969989d8d43,2026-09-12 13:03:47.107054,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,310000,003000,000000,000000,LGAAP1,NFTBAL,USD,-20000.000000000000,0E-12,-20000.000000000000
2,e0161602-3a56-4b0c-9165-2a3bfb06f733,2026-09-12 13:03:47.107108,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USTRD1,990101,002000,000000,000000,LGAAP1,NFTBAL,USD,0E-12,-65000.000000000000,65000.000000000000
3,6a1980e4-59fd-4d6b-a905-b745b80468f4,2026-09-12 13:03:47.107030,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,CAMKTS,CATO01,CATRD1,990201,990001,100001,990003,LGAAP1,NFTBAL,USD,0E-12,-38250.000000000000,38250.000000000000
4,0a6a3f0c-b9cf-49ba-8875-14bd28cd23af,2026-09-12 13:03:47.106993,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,CAMKTS,CATO01,CATRD1,,,100001,,LGAAP1,NFTBAL,USD,-38250.000000000000,0E-12,-38250.000000000000
5,56113b7e-582f-41d9-98ed-8575fceda4ab,2026-09-12 13:03:47.107097,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USTRD1,210000,002000,000000,000000,LGAAP1,NFTBAL,USD,-65000.000000000000,0E-12,-65000.000000000000
6,bd5598a3-6a7c-40d0-a3bc-42d2ca52f930,2026-09-12 13:03:47.107065,694af9a4-9c03-419f-a9d6-705ab094677d,a48af47a-994d-456d-8b7d-16bcfafe9086,2026-03-31,USMKTS,USNY01,USFIN1,4100000,004000,000000,000000,LGAAP1,NFTBAL,USD,-25000.000000000000,0E-12,-25000.000000000000


In [17]:
spark.stop()